# NB 01 - Setup: Database, Schemas, Tables & Sample Data

Sets up the full foundation: database, schemas, warehouses, roles, all RAW/PROCESSED/RESULTS/VECTORS tables, and loads the Manapakkam fire sample data.

**Run all cells top to bottom.**

## 01_SETUP_DATABASE.SQL

Insurance Claims Agentic AI - Snowflake GCC Hackathon
Database, Schemas, Warehouses, Roles, Grants

In [ ]:
%%sql -r dataframe_1
-- Use ACCOUNTADMIN for initial setup
USE ROLE ACCOUNTADMIN;

## DATABASE

In [ ]:
%%sql -r dataframe_2
CREATE DATABASE IF NOT EXISTS INSURANCE_DB
    COMMENT = 'Insurance Claims Agentic AI - Customer 360 + NBA';

USE DATABASE INSURANCE_DB;

## SCHEMAS (Layered Architecture)

In [ ]:
%%sql -r dataframe_3
CREATE SCHEMA IF NOT EXISTS RAW
    COMMENT = 'Source of truth - ingested structured + unstructured data';

CREATE SCHEMA IF NOT EXISTS PROCESSED
    COMMENT = 'Enriched data - agent outputs, Customer 360, NBA';

CREATE SCHEMA IF NOT EXISTS RESULTS
    COMMENT = 'Final decisions - resolutions, underwriting, audit trail';

CREATE SCHEMA IF NOT EXISTS VECTORS
    COMMENT = 'RAG embeddings - document chunks, fraud patterns';

CREATE SCHEMA IF NOT EXISTS FEATURES
    COMMENT = 'ML Feature Store - numerical features for model training/serving';

CREATE SCHEMA IF NOT EXISTS STAGES
    COMMENT = 'Internal stages for semantic models, documents, knowledge bases';

## WAREHOUSES (Cost-Optimized)

In [ ]:
%%sql -r dataframe_4
-- Heavy AI/LLM processing (agents, Cortex calls)
CREATE WAREHOUSE IF NOT EXISTS AGENT_WH
    WAREHOUSE_SIZE = 'MEDIUM'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    MIN_CLUSTER_COUNT = 1
    MAX_CLUSTER_COUNT = 3
    SCALING_POLICY = 'STANDARD'
    COMMENT = 'Agent pipeline processing - Cortex LLM/Analyst/Search calls';

-- Lightweight scheduled scans (churn daily scan, notifications)
CREATE WAREHOUSE IF NOT EXISTS EVAL_WH
    WAREHOUSE_SIZE = 'X-SMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    COMMENT = 'Lightweight evaluation tasks - daily scans, monitoring';

-- Data ingestion workloads
CREATE WAREHOUSE IF NOT EXISTS INGEST_WH
    WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    COMMENT = 'Data ingestion - Snowpipe, Stream consumption';

## ROLES (Least Privilege RBAC)

In [ ]:
%%sql -r dataframe_5
-- Application service role (agents, orchestrator)
CREATE ROLE IF NOT EXISTS INSURANCE_APP_ROLE
    COMMENT = 'Service role for agent pipeline execution';

-- Claims adjuster (read decisions, view 360)
CREATE ROLE IF NOT EXISTS INSURANCE_ADJUSTER_ROLE
    COMMENT = 'Claims adjusters - view decisions and Customer 360';

-- Underwriter (underwriting decisions)
CREATE ROLE IF NOT EXISTS INSURANCE_UNDERWRITER_ROLE
    COMMENT = 'Underwriters - view/action underwriting decisions';

-- Retention analyst (churn + NBA)
CREATE ROLE IF NOT EXISTS INSURANCE_RETENTION_ROLE
    COMMENT = 'Retention analysts - view churn alerts and NBA';

-- Admin role (full access)
CREATE ROLE IF NOT EXISTS INSURANCE_ADMIN_ROLE
    COMMENT = 'Admin - full database management';

## ROLE HIERARCHY

In [ ]:
%%sql -r dataframe_6
GRANT ROLE INSURANCE_APP_ROLE TO ROLE INSURANCE_ADMIN_ROLE;
GRANT ROLE INSURANCE_ADJUSTER_ROLE TO ROLE INSURANCE_ADMIN_ROLE;
GRANT ROLE INSURANCE_UNDERWRITER_ROLE TO ROLE INSURANCE_ADMIN_ROLE;
GRANT ROLE INSURANCE_RETENTION_ROLE TO ROLE INSURANCE_ADMIN_ROLE;
GRANT ROLE INSURANCE_ADMIN_ROLE TO ROLE SYSADMIN;

## GRANTS - DATABASE & SCHEMA

In [ ]:
%%sql -r dataframe_7
-- App role needs full CRUD on all schemas
GRANT USAGE ON DATABASE INSURANCE_DB TO ROLE INSURANCE_APP_ROLE;
GRANT USAGE ON ALL SCHEMAS IN DATABASE INSURANCE_DB TO ROLE INSURANCE_APP_ROLE;
GRANT ALL PRIVILEGES ON SCHEMA INSURANCE_DB.RAW TO ROLE INSURANCE_APP_ROLE;
GRANT ALL PRIVILEGES ON SCHEMA INSURANCE_DB.PROCESSED TO ROLE INSURANCE_APP_ROLE;
GRANT ALL PRIVILEGES ON SCHEMA INSURANCE_DB.RESULTS TO ROLE INSURANCE_APP_ROLE;
GRANT ALL PRIVILEGES ON SCHEMA INSURANCE_DB.VECTORS TO ROLE INSURANCE_APP_ROLE;
GRANT ALL PRIVILEGES ON SCHEMA INSURANCE_DB.FEATURES TO ROLE INSURANCE_APP_ROLE;
GRANT ALL PRIVILEGES ON SCHEMA INSURANCE_DB.STAGES TO ROLE INSURANCE_APP_ROLE;

-- Adjuster: read RESULTS + PROCESSED
GRANT USAGE ON DATABASE INSURANCE_DB TO ROLE INSURANCE_ADJUSTER_ROLE;
GRANT USAGE ON SCHEMA INSURANCE_DB.RESULTS TO ROLE INSURANCE_ADJUSTER_ROLE;
GRANT USAGE ON SCHEMA INSURANCE_DB.PROCESSED TO ROLE INSURANCE_ADJUSTER_ROLE;
GRANT SELECT ON ALL TABLES IN SCHEMA INSURANCE_DB.RESULTS TO ROLE INSURANCE_ADJUSTER_ROLE;
GRANT SELECT ON ALL TABLES IN SCHEMA INSURANCE_DB.PROCESSED TO ROLE INSURANCE_ADJUSTER_ROLE;

-- Underwriter: read RESULTS + RAW.APPLICATIONS
GRANT USAGE ON DATABASE INSURANCE_DB TO ROLE INSURANCE_UNDERWRITER_ROLE;
GRANT USAGE ON SCHEMA INSURANCE_DB.RESULTS TO ROLE INSURANCE_UNDERWRITER_ROLE;
GRANT USAGE ON SCHEMA INSURANCE_DB.RAW TO ROLE INSURANCE_UNDERWRITER_ROLE;
GRANT SELECT ON ALL TABLES IN SCHEMA INSURANCE_DB.RESULTS TO ROLE INSURANCE_UNDERWRITER_ROLE;

-- Retention: read PROCESSED (churn, NBA)
GRANT USAGE ON DATABASE INSURANCE_DB TO ROLE INSURANCE_RETENTION_ROLE;
GRANT USAGE ON SCHEMA INSURANCE_DB.PROCESSED TO ROLE INSURANCE_RETENTION_ROLE;
GRANT SELECT ON ALL TABLES IN SCHEMA INSURANCE_DB.PROCESSED TO ROLE INSURANCE_RETENTION_ROLE;

## GRANTS - WAREHOUSES

In [ ]:
%%sql -r dataframe_8
GRANT USAGE ON WAREHOUSE AGENT_WH TO ROLE INSURANCE_APP_ROLE;
GRANT USAGE ON WAREHOUSE EVAL_WH TO ROLE INSURANCE_APP_ROLE;
GRANT USAGE ON WAREHOUSE INGEST_WH TO ROLE INSURANCE_APP_ROLE;
GRANT USAGE ON WAREHOUSE AGENT_WH TO ROLE INSURANCE_ADJUSTER_ROLE;
GRANT USAGE ON WAREHOUSE EVAL_WH TO ROLE INSURANCE_RETENTION_ROLE;
GRANT USAGE ON WAREHOUSE AGENT_WH TO ROLE INSURANCE_UNDERWRITER_ROLE;

## CORTEX ACCESS GRANTS

In [ ]:
%%sql -r dataframe_9
-- Grant Cortex function access to app role
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE INSURANCE_APP_ROLE;

## INTERNAL STAGES (for semantic models + documents)

In [ ]:
%%sql -r dataframe_10
USE SCHEMA INSURANCE_DB.STAGES;

CREATE STAGE IF NOT EXISTS SEMANTIC_MODELS_STAGE
    COMMENT = 'Cortex Analyst semantic model YAML files';

CREATE STAGE IF NOT EXISTS DOCUMENTS_STAGE
    COMMENT = 'Uploaded claim documents (PDFs, FIRs, invoices)';

CREATE STAGE IF NOT EXISTS KNOWLEDGE_BASE_STAGE
    COMMENT = 'RAG knowledge base documents (underwriting guides, playbooks)';

## NOTIFICATION INTEGRATION (for stakeholder alerts)

In [ ]:
%%sql -r dataframe_11
CREATE NOTIFICATION INTEGRATION IF NOT EXISTS CLAIMS_EMAIL_NOTIFICATION
    TYPE = EMAIL
    ENABLED = TRUE
    ALLOWED_RECIPIENTS = ('raamamoorthi.s@idp.com')
    COMMENT = 'Email notifications for claim decisions and escalations';

## NETWORK RULE + EXTERNAL ACCESS (for credit bureau API if needed)

> CREATE NETWORK RULE credit_bureau_rule  
> TYPE = HOST_PORT  
> MODE = EGRESS  
> VALUE_LIST = ('api.creditbureau.com:443');  
>   
> CREATE EXTERNAL ACCESS INTEGRATION credit_bureau_integration  
> ALLOWED_NETWORK_RULES = (credit_bureau_rule)  
> ENABLED = TRUE;

## TAGS (Data Governance)

In [ ]:
%%sql -r dataframe_12
CREATE TAG IF NOT EXISTS INSURANCE_DB.RAW.PII_TAG
    COMMENT = 'Marks columns containing Personally Identifiable Information';

CREATE TAG IF NOT EXISTS INSURANCE_DB.RAW.SENSITIVITY_TAG
    ALLOWED_VALUES 'LOW', 'MEDIUM', 'HIGH', 'CRITICAL'
    COMMENT = 'Data sensitivity classification';

## DONE - Setup Complete

> Next: Run 02_raw_tables.sql to create source tables and load sample data

## 02_RAW_TABLES.SQL

Insurance Claims Agentic AI - Snowflake GCC Hackathon
RAW Schema Tables + Manapakkam Fire Sample Data

In [ ]:
%%sql -r dataframe_13
USE ROLE INSURANCE_APP_ROLE;
USE DATABASE INSURANCE_DB;
USE WAREHOUSE INGEST_WH;

## RAW.CUSTOMERS - Customer Master

In [ ]:
%%sql -r dataframe_14
CREATE OR REPLACE TABLE RAW.CUSTOMERS (
    customer_id         VARCHAR(50) NOT NULL,
    first_name          VARCHAR(100),
    last_name           VARCHAR(100),
    email               VARCHAR(200),
    phone               VARCHAR(20),
    date_of_birth       DATE,
    address             VARCHAR(500),
    city                VARCHAR(100),
    state_province      VARCHAR(50),
    pin_code            VARCHAR(10),
    customer_segment    VARCHAR(20),       -- HNW / STANDARD / SME
    lifetime_value_score FLOAT,            -- Normalized 0-1
    onboarding_date     DATE,
    kyc_status          VARCHAR(20),       -- VERIFIED / PENDING / REJECTED
    preferred_channel   VARCHAR(20),       -- CALL / EMAIL / SMS / PORTAL
    created_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    updated_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    CONSTRAINT pk_customers PRIMARY KEY (customer_id)
);

## RAW.POLICIES - Multi-LOB Policy Master

In [ ]:
%%sql -r dataframe_15
CREATE OR REPLACE TABLE RAW.POLICIES (
    policy_id           VARCHAR(50) NOT NULL,
    customer_id         VARCHAR(50) NOT NULL,
    lob_type            VARCHAR(30) NOT NULL,  -- AUTO / PROPERTY / WORKERS_COMP
    policy_status       VARCHAR(20),           -- ACTIVE / EXPIRED / CANCELLED / LAPSED
    start_date          DATE,
    end_date            DATE,
    renewal_date        DATE,
    premium_annual      FLOAT,
    coverage_limit      FLOAT,
    deductible          FLOAT,
    exclusions          VARIANT,               -- JSON array of exclusion clauses
    asset_details       VARIANT,               -- JSON: vehicle/property/employer info
    underwriting_tier   VARCHAR(20),           -- LOW / MEDIUM / HIGH
    geographic_zone     VARCHAR(50),           -- CHENNAI_FLOOD / CHENNAI_FIRE / etc.
    provider_network    VARCHAR(20),           -- PREFERRED / STANDARD / ANY
    created_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    updated_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    CONSTRAINT pk_policies PRIMARY KEY (policy_id)
);

## RAW.CLAIMS_LANDING - Claim Intake (Streamlit submits here)

In [ ]:
%%sql -r dataframe_16
CREATE OR REPLACE TABLE RAW.CLAIMS_LANDING (
    claim_id            VARCHAR(50) NOT NULL DEFAULT UUID_STRING(),
    policy_id           VARCHAR(50),
    customer_id         VARCHAR(50),
    claim_text          VARCHAR(10000),        -- Natural language claim description
    incident_date       DATE,
    submission_date     TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    claimed_amount      FLOAT,
    lob_type            VARCHAR(30),
    claim_status        VARCHAR(30) DEFAULT 'SUBMITTED',  -- State machine
    incident_location   VARCHAR(500),
    supporting_docs     VARIANT,               -- JSON array of document references
    metadata            VARIANT,               -- Additional form fields
    created_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    CONSTRAINT pk_claims_landing PRIMARY KEY (claim_id)
);

## RAW.PROVIDERS - Hospitals, Garages, Contractors

In [ ]:
%%sql -r dataframe_17
CREATE OR REPLACE TABLE RAW.PROVIDERS (
    provider_id         VARCHAR(50) NOT NULL,
    provider_name       VARCHAR(200),
    provider_type       VARCHAR(50),           -- HOSPITAL / GARAGE / CONTRACTOR / CLINIC
    lob_type            VARCHAR(30),
    network_status      VARCHAR(20),           -- IN_NETWORK / OUT_NETWORK
    risk_flag           BOOLEAN DEFAULT FALSE,
    avg_billing_amount  FLOAT,
    total_claims_served INT,
    fraud_flag_count    INT DEFAULT 0,
    location_city       VARCHAR(100),
    location_state      VARCHAR(50),
    license_number      VARCHAR(100),
    created_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    CONSTRAINT pk_providers PRIMARY KEY (provider_id)
);

## RAW.FRAUD_INDICATORS - Historical Confirmed Fraud Patterns

In [ ]:
%%sql -r dataframe_18
CREATE OR REPLACE TABLE RAW.FRAUD_INDICATORS (
    indicator_id        VARCHAR(50) NOT NULL,
    pattern_type        VARCHAR(100),          -- DUPLICATE / INFLATED / STAGED / TIMING / IDENTITY
    pattern_description VARCHAR(5000),         -- Detailed description for RAG
    lob_type            VARCHAR(30),
    severity            VARCHAR(20),           -- LOW / MEDIUM / HIGH / CRITICAL
    detection_signals   VARIANT,              -- JSON: key features that trigger this pattern
    example_narrative   VARCHAR(5000),         -- Example claim text matching this pattern
    confirmed_cases     INT,
    last_seen_date      DATE,
    created_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    CONSTRAINT pk_fraud_indicators PRIMARY KEY (indicator_id)
);

## RAW.INTERACTIONS - Customer Interactions (Calls, Emails, Chats)

In [ ]:
%%sql -r dataframe_19
CREATE OR REPLACE TABLE RAW.INTERACTIONS (
    interaction_id      VARCHAR(50) NOT NULL,
    customer_id         VARCHAR(50),
    channel             VARCHAR(20),           -- CALL / EMAIL / CHAT / PORTAL
    interaction_date    TIMESTAMP_NTZ,
    transcript_text     VARCHAR(50000),        -- Full call transcript / email body
    direction           VARCHAR(10),           -- INBOUND / OUTBOUND
    topic               VARCHAR(100),
    resolution_status   VARCHAR(20),           -- RESOLVED / PENDING / ESCALATED
    agent_id            VARCHAR(50),
    duration_seconds    INT,
    nps_score           INT,                   -- 1-10 if collected
    created_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    CONSTRAINT pk_interactions PRIMARY KEY (interaction_id)
);

## RAW.PAYMENTS - Premium Payment History

In [ ]:
%%sql -r dataframe_20
CREATE OR REPLACE TABLE RAW.PAYMENTS (
    payment_id          VARCHAR(50) NOT NULL,
    policy_id           VARCHAR(50),
    customer_id         VARCHAR(50),
    payment_date        DATE,
    due_date            DATE,
    amount              FLOAT,
    payment_status      VARCHAR(20),           -- PAID / OVERDUE / MISSED / PARTIAL
    payment_method      VARCHAR(30),           -- AUTO_DEBIT / UPI / NEFT / CARD / CASH
    days_delayed        INT DEFAULT 0,
    created_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    CONSTRAINT pk_payments PRIMARY KEY (payment_id)
);

## RAW.APPLICATIONS - Policy Applications (Underwriting)

In [ ]:
%%sql -r dataframe_21
CREATE OR REPLACE TABLE RAW.APPLICATIONS (
    application_id      VARCHAR(50) NOT NULL DEFAULT UUID_STRING(),
    applicant_name      VARCHAR(200),
    applicant_email     VARCHAR(200),
    existing_customer_id VARCHAR(50),          -- NULL if new applicant
    lob_type            VARCHAR(30),
    coverage_requested  FLOAT,
    asset_details       VARIANT,              -- JSON: vehicle/property/employer
    self_declared_history VARIANT,            -- JSON: prior claims, health conditions
    credit_score        INT,
    geographic_zone     VARCHAR(50),
    employer_name       VARCHAR(200),          -- For Workers' Comp
    submission_date     TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    status              VARCHAR(20) DEFAULT 'PENDING',
    CONSTRAINT pk_applications PRIMARY KEY (application_id)
);

## RAW.DOCUMENTS - Unstructured Document Metadata

In [ ]:
%%sql -r dataframe_22
CREATE OR REPLACE TABLE RAW.DOCUMENTS (
    document_id         VARCHAR(50) NOT NULL DEFAULT UUID_STRING(),
    claim_id            VARCHAR(50),
    document_type       VARCHAR(50),           -- POLICE_FIR / INVOICE / MEDICAL_REPORT / FIRE_REPORT / PHOTO
    file_name           VARCHAR(200),
    stage_path          VARCHAR(500),          -- @DOCUMENTS_STAGE/path
    lob_type            VARCHAR(30),
    upload_date         TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    extracted_text      VARCHAR(100000),       -- Parsed/OCR text content
    extraction_status   VARCHAR(20) DEFAULT 'PENDING', -- PENDING / EXTRACTED / FAILED
    page_count          INT,
    file_size_bytes     INT,
    CONSTRAINT pk_documents PRIMARY KEY (document_id)
);

## RAW.ACTUARIAL_TABLES - Premium Calculation Reference

In [ ]:
%%sql -r dataframe_23
CREATE OR REPLACE TABLE RAW.ACTUARIAL_TABLES (
    table_id            VARCHAR(50) NOT NULL,
    lob_type            VARCHAR(30),
    risk_tier           VARCHAR(20),
    geographic_zone     VARCHAR(50),
    base_rate           FLOAT,                 -- Base premium rate
    risk_multiplier     FLOAT,
    geographic_factor   FLOAT,
    claims_history_factor FLOAT,
    effective_date      DATE,
    expiry_date         DATE,
    CONSTRAINT pk_actuarial PRIMARY KEY (table_id)
);

## RAW.UNDERWRITING_GUIDELINES - Knowledge Base for RAG

In [ ]:
%%sql -r dataframe_24
CREATE OR REPLACE TABLE RAW.UNDERWRITING_GUIDELINES (
    guideline_id        VARCHAR(50) NOT NULL,
    lob_type            VARCHAR(30),
    section_name        VARCHAR(200),
    content             VARCHAR(10000),        -- Full guideline text
    risk_level          VARCHAR(20),           -- What risk level this applies to
    regulatory_reference VARCHAR(200),
    effective_date      DATE,
    status              VARCHAR(20) DEFAULT 'CURRENT', -- CURRENT / SUPERSEDED
    CONSTRAINT pk_guidelines PRIMARY KEY (guideline_id)
);

## RAW.RETENTION_PLAYBOOKS - Churn Control Knowledge Base

In [ ]:
%%sql -r dataframe_25
CREATE OR REPLACE TABLE RAW.RETENTION_PLAYBOOKS (
    playbook_id         VARCHAR(50) NOT NULL,
    customer_segment    VARCHAR(20),           -- HNW / STANDARD / SME
    root_cause          VARCHAR(100),          -- price_sensitivity / poor_claims_exp / etc.
    playbook_name       VARCHAR(200),
    content             VARCHAR(10000),        -- Full playbook text
    recommended_actions VARIANT,              -- JSON array of actions
    success_rate        FLOAT,                 -- Historical success rate
    applicable_lob      VARCHAR(30),
    CONSTRAINT pk_playbooks PRIMARY KEY (playbook_id)
);

## PROCESSED SCHEMA TABLES

In [ ]:
%%sql -r dataframe_26
USE SCHEMA PROCESSED;

-- Claim state tracking (state machine)
CREATE OR REPLACE TABLE PROCESSED.CLAIM_STATE (
    claim_id            VARCHAR(50) NOT NULL,
    current_state       VARCHAR(30),           -- SUBMITTED/INTAKE/VALIDATION/FRAUD/ASSESSMENT/RESOLUTION/COMPLETED/FAILED
    intake_output       VARIANT,              -- JSON from intake agent
    validation_output   VARIANT,              -- JSON from validation agent
    fraud_output        VARIANT,              -- JSON from fraud agent
    assessment_output   VARIANT,              -- JSON from assessment agent
    resolution_output   VARIANT,              -- JSON from resolution agent
    started_at          TIMESTAMP_NTZ,
    completed_at        TIMESTAMP_NTZ,
    error_message       VARCHAR(2000),
    retry_count         INT DEFAULT 0,
    CONSTRAINT pk_claim_state PRIMARY KEY (claim_id)
);

-- Customer 360 (materialized daily)
CREATE OR REPLACE TABLE PROCESSED.FCT_CUSTOMER_360 (
    customer_id                 VARCHAR(50) NOT NULL,
    -- Identity & Demographics
    customer_segment            VARCHAR(20),
    lifetime_value_score        FLOAT,
    tenure_months               INT,
    -- Policy Portfolio (structured)
    active_policy_count         INT,
    total_annual_premium        FLOAT,
    lob_diversity               INT,
    days_to_nearest_renewal     INT,
    coverage_adequacy_ratio     FLOAT,
    lapse_count_historical      INT,
    policy_ids_array            VARIANT,       -- JSON array of active policy IDs
    -- Claims History (structured)
    claim_count_12m             INT,
    claim_count_lifetime        INT,
    claims_approved_ratio       FLOAT,
    avg_claim_amount            FLOAT,
    max_claim_amount            FLOAT,
    total_amount_paid           FLOAT,
    avg_resolution_time_days    FLOAT,
    open_claims_count           INT,
    fraud_flag_count            INT,
    loss_ratio                  FLOAT,
    claim_acceleration_ratio    FLOAT,
    -- Payment Behavior (structured)
    payment_delay_avg_days      FLOAT,
    missed_payments_12m         INT,
    payment_regularity_score    FLOAT,
    last_payment_days_ago       INT,
    auto_pay_enrolled           BOOLEAN,
    -- Interaction Signals (Cortex LLM extracted from unstructured)
    sentiment_score_last_30d    FLOAT,
    sentiment_trend             FLOAT,
    frustration_level           FLOAT,
    escalation_flag             BOOLEAN,
    complaint_count_90d         INT,
    nps_score_latest            INT,
    intent_cancel_detected      BOOLEAN,
    competitor_mentions_count   INT,
    channel_preference          VARCHAR(20),
    last_interaction_days_ago   INT,
    digital_engagement_trend    FLOAT,
    -- Fraud Narrative Signals (Cortex LLM extracted)
    emotional_manipulation_flag BOOLEAN,
    inconsistency_score         FLOAT,
    vagueness_score             FLOAT,
    -- Embedding Features (Cortex Embed)
    fraud_similarity_score      FLOAT,
    claim_cluster_id            INT,
    provider_anomaly_score      FLOAT,
    -- Derived / Cross-domain
    renewal_risk_signal         BOOLEAN,
    complaint_rate_per_claim    FLOAT,
    lifetime_loss_ratio         FLOAT,
    high_value_at_risk          BOOLEAN,
    geographic_risk_score       FLOAT,
    -- ML Model Scores (Snowpark ML UDFs)
    churn_propensity_score      FLOAT,
    churn_time_to_event_days    INT,
    fraud_propensity_score      FLOAT,
    underwriting_risk_tier      VARCHAR(20),
    expected_loss_ratio         FLOAT,
    cross_sell_propensity       FLOAT,
    recommended_lob             VARCHAR(30),
    -- Next Best Action
    next_best_action            VARCHAR(50),
    nba_rationale               VARCHAR(2000),
    nba_priority                INT,
    -- Metadata
    last_refreshed_at           TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    CONSTRAINT pk_customer_360 PRIMARY KEY (customer_id)
);

-- View wrapper
CREATE OR REPLACE VIEW PROCESSED.FCT_CUSTOMER_360_VIEW AS
SELECT * FROM PROCESSED.FCT_CUSTOMER_360;

-- Churn alerts
CREATE OR REPLACE TABLE PROCESSED.CHURN_ALERTS (
    alert_id            VARCHAR(50) DEFAULT UUID_STRING(),
    customer_id         VARCHAR(50),
    alert_date          DATE DEFAULT CURRENT_DATE(),
    churn_propensity    FLOAT,
    sentiment_score     FLOAT,
    complaint_count     INT,
    trigger_reason      VARCHAR(200),
    priority            INT,
    status              VARCHAR(20) DEFAULT 'NEW',
    root_cause          VARCHAR(100),
    root_cause_confidence FLOAT,
    created_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- Next Best Actions
CREATE OR REPLACE TABLE PROCESSED.NEXT_BEST_ACTIONS (
    nba_id              VARCHAR(50) DEFAULT UUID_STRING(),
    customer_id         VARCHAR(50),
    action_type         VARCHAR(50),
    offer_details       VARCHAR(2000),
    channel             VARCHAR(20),
    timing              VARCHAR(50),
    priority            INT,
    expected_success_rate FLOAT,
    root_cause          VARCHAR(100),
    rationale           VARCHAR(2000),
    status              VARCHAR(20) DEFAULT 'PENDING',
    created_date        DATE DEFAULT CURRENT_DATE(),
    expiry_date         DATE,
    outcome             VARCHAR(20),
    created_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

## RESULTS SCHEMA TABLES

In [ ]:
%%sql -r dataframe_27
USE SCHEMA RESULTS;

CREATE OR REPLACE TABLE RESULTS.RESOLUTIONS (
    resolution_id       VARCHAR(50) DEFAULT UUID_STRING(),
    claim_id            VARCHAR(50),
    customer_id         VARCHAR(50),
    policy_id           VARCHAR(50),
    lob_type            VARCHAR(30),
    decision            VARCHAR(30),          -- APPROVED / DENIED / PARTIALLY_APPROVED / REFERRED
    settlement_amount   FLOAT,
    claimed_amount      FLOAT,
    fraud_risk_level    VARCHAR(20),
    fraud_score         FLOAT,
    reasoning_summary   VARCHAR(5000),
    confidence_score    FLOAT,
    next_steps          VARCHAR(2000),
    processing_time_ms  INT,
    decided_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

CREATE OR REPLACE TABLE RESULTS.UNDERWRITING_DECISIONS (
    decision_id         VARCHAR(50) DEFAULT UUID_STRING(),
    application_id      VARCHAR(50),
    customer_id         VARCHAR(50),
    lob_type            VARCHAR(30),
    risk_tier           VARCHAR(20),
    confidence          FLOAT,
    recommended_premium FLOAT,
    exclusions          VARIANT,
    terms_text          VARCHAR(5000),
    auto_approved       BOOLEAN,
    rationale           VARCHAR(2000),
    decided_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

CREATE OR REPLACE TABLE RESULTS.AUDIT_LOG (
    audit_id            VARCHAR(50) DEFAULT UUID_STRING(),
    flow_type           VARCHAR(30),          -- CLAIMS / UNDERWRITING / CHURN
    reference_id        VARCHAR(50),
    agent_name          VARCHAR(50),
    step_number         INT,
    input_payload       VARIANT,
    output_payload      VARIANT,
    cortex_module_used  VARCHAR(50),
    model_used          VARCHAR(50),
    tokens_input        INT,
    tokens_output       INT,
    latency_ms          INT,
    status              VARCHAR(20),
    error_message       VARCHAR(2000),
    created_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

## VECTORS SCHEMA TABLES

In [ ]:
%%sql -r dataframe_28
USE SCHEMA VECTORS;

CREATE OR REPLACE TABLE VECTORS.DOCUMENT_CHUNKS (
    chunk_id            VARCHAR(50) DEFAULT UUID_STRING(),
    document_id         VARCHAR(50),
    claim_id            VARCHAR(50),
    chunk_index         INT,
    chunk_text          VARCHAR(8000),
    chunk_embedding     VECTOR(FLOAT, 768),   -- Snowflake native VECTOR type
    document_type       VARCHAR(50),
    lob_type            VARCHAR(30),
    metadata            VARIANT,
    created_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

CREATE OR REPLACE TABLE VECTORS.FRAUD_PATTERN_EMBEDDINGS (
    pattern_id          VARCHAR(50),
    pattern_type        VARCHAR(100),
    pattern_description VARCHAR(5000),
    pattern_embedding   VECTOR(FLOAT, 768),
    severity            VARCHAR(20),
    lob_type            VARCHAR(30),
    created_at          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

## SAMPLE DATA: MANAPAKKAM FIRE SCENARIO

In [ ]:
%%sql -r dataframe_29
USE SCHEMA RAW;

-- Insert Customers (including HNW client from problem statement)
INSERT INTO RAW.CUSTOMERS VALUES
('CUST001', 'Rajesh', 'Krishnamurthy', 'rajesh.k@techfirm.in', '+91-9876543210', '1975-03-15',
 '42 Tech Park Road, Manapakkam', 'Chennai', 'Tamil Nadu', '600089', 'HNW', 0.92,
 '2015-01-10', 'VERIFIED', 'CALL', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()),

('CUST002', 'Priya', 'Venkatesh', 'priya.v@startup.io', '+91-9876543211', '1988-07-22',
 '15 Innovation Drive, Manapakkam', 'Chennai', 'Tamil Nadu', '600089', 'SME', 0.65,
 '2019-06-01', 'VERIFIED', 'EMAIL', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()),

('CUST003', 'Arjun', 'Sundaram', 'arjun.s@gmail.com', '+91-9876543212', '1992-11-08',
 '78 Velachery Main Road', 'Chennai', 'Tamil Nadu', '600042', 'STANDARD', 0.35,
 '2022-03-15', 'VERIFIED', 'SMS', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()),

('CUST004', 'Meenakshi', 'Rajan', 'meena.r@manufacturing.co.in', '+91-9876543213', '1970-05-30',
 '101 Industrial Estate, Manapakkam', 'Chennai', 'Tamil Nadu', '600089', 'HNW', 0.88,
 '2012-09-01', 'VERIFIED', 'CALL', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()),

('CUST005', 'Vikram', 'Patel', 'vikram.p@fakemail.com', '+91-9876543214', '1995-01-01',
 '99 Unknown Street', 'Chennai', 'Tamil Nadu', '600001', 'STANDARD', 0.15,
 '2024-11-01', 'VERIFIED', 'PORTAL', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP());

-- Insert Policies (Multi-LOB for HNW clients)
-- Note: Using INSERT INTO ... SELECT because PARSE_JSON() cannot be used in VALUES clause

-- CUST001 (HNW) - Has Auto + Property + Workers Comp
INSERT INTO RAW.POLICIES SELECT
 'POL-AUTO-001', 'CUST001', 'AUTO', 'ACTIVE', '2024-01-01'::DATE, '2025-12-31'::DATE, '2025-12-31'::DATE,
 45000, 2500000, 25000, PARSE_JSON('["racing", "drunk_driving"]'),
 PARSE_JSON('{"vehicles": [{"make": "BMW", "model": "X5", "year": 2023, "reg": "TN01AB1234"}, {"make": "Mercedes", "model": "E-Class", "year": 2022, "reg": "TN01CD5678"}]}'),
 'LOW', 'CHENNAI_MANAPAKKAM', 'PREFERRED', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP();

INSERT INTO RAW.POLICIES SELECT
 'POL-PROP-001', 'CUST001', 'PROPERTY', 'ACTIVE', '2024-01-01'::DATE, '2025-12-31'::DATE, '2025-12-31'::DATE,
 95000, 50000000, 500000, PARSE_JSON('["war", "nuclear"]'),
 PARSE_JSON('{"property_type": "commercial_office", "area_sqft": 15000, "floors": 3, "address": "Block B, Tech Park, Manapakkam"}'),
 'LOW', 'CHENNAI_MANAPAKKAM', 'PREFERRED', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP();

INSERT INTO RAW.POLICIES SELECT
 'POL-WC-001', 'CUST001', 'WORKERS_COMP', 'ACTIVE', '2024-01-01'::DATE, '2025-12-31'::DATE, '2025-12-31'::DATE,
 40000, 10000000, 50000, PARSE_JSON('["pre_existing_conditions"]'),
 PARSE_JSON('{"employer": "TechFirm Solutions Pvt Ltd", "employee_count": 250, "industry": "IT_Services", "hazard_class": "LOW"}'),
 'LOW', 'CHENNAI_MANAPAKKAM', 'PREFERRED', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP();

-- CUST002 (SME) - Property only
INSERT INTO RAW.POLICIES SELECT
 'POL-PROP-002', 'CUST002', 'PROPERTY', 'ACTIVE', '2024-06-01'::DATE, '2025-05-31'::DATE, '2025-05-31'::DATE,
 25000, 5000000, 100000, PARSE_JSON('["earthquake"]'),
 PARSE_JSON('{"property_type": "office_startup", "area_sqft": 2000, "floors": 1, "address": "15 Innovation Drive, Manapakkam"}'),
 'MEDIUM', 'CHENNAI_MANAPAKKAM', 'STANDARD', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP();

-- CUST003 (Standard) - Auto only
INSERT INTO RAW.POLICIES SELECT
 'POL-AUTO-003', 'CUST003', 'AUTO', 'ACTIVE', '2024-03-01'::DATE, '2025-02-28'::DATE, '2025-02-28'::DATE,
 12000, 500000, 10000, PARSE_JSON('["commercial_use"]'),
 PARSE_JSON('{"vehicles": [{"make": "Hyundai", "model": "i20", "year": 2021, "reg": "TN07EF9012"}]}'),
 'MEDIUM', 'CHENNAI_VELACHERY', 'STANDARD', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP();

-- CUST004 (HNW) - Property + Workers Comp
INSERT INTO RAW.POLICIES SELECT
 'POL-PROP-004', 'CUST004', 'PROPERTY', 'ACTIVE', '2024-01-01'::DATE, '2025-12-31'::DATE, '2025-12-31'::DATE,
 120000, 100000000, 1000000, PARSE_JSON('["terrorism"]'),
 PARSE_JSON('{"property_type": "manufacturing_unit", "area_sqft": 50000, "floors": 2, "address": "101 Industrial Estate, Manapakkam"}'),
 'LOW', 'CHENNAI_MANAPAKKAM', 'PREFERRED', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP();

INSERT INTO RAW.POLICIES SELECT
 'POL-WC-004', 'CUST004', 'WORKERS_COMP', 'ACTIVE', '2024-01-01'::DATE, '2025-12-31'::DATE, '2025-12-31'::DATE,
 60000, 20000000, 100000, PARSE_JSON('["pre_existing_conditions"]'),
 PARSE_JSON('{"employer": "Chennai Manufacturing Corp", "employee_count": 500, "industry": "Manufacturing", "hazard_class": "MEDIUM"}'),
 'MEDIUM', 'CHENNAI_MANAPAKKAM', 'PREFERRED', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP();

-- CUST005 (Suspicious - new customer, no history) - Auto
INSERT INTO RAW.POLICIES SELECT
 'POL-AUTO-005', 'CUST005', 'AUTO', 'ACTIVE', '2024-11-01'::DATE, '2025-10-31'::DATE, '2025-10-31'::DATE,
 8000, 300000, 5000, PARSE_JSON('["racing"]'),
 PARSE_JSON('{"vehicles": [{"make": "Maruti", "model": "Alto", "year": 2018, "reg": "TN99ZZ0001"}]}'),
 'HIGH', 'CHENNAI_GENERAL', 'STANDARD', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP();

-- Insert Claims (Manapakkam Fire - multiple simultaneous claims)
INSERT INTO RAW.CLAIMS_LANDING (claim_id, policy_id, customer_id, claim_text, incident_date, claimed_amount, lob_type, incident_location) VALUES

-- CUST001: Legitimate multi-LOB disaster claims (HNW)
('CLM-001-AUTO', 'POL-AUTO-001', 'CUST001',
 'Massive fire at Manapakkam Tech Park on 15th Jan 2025. My BMW X5 (TN01AB1234) and Mercedes E-Class (TN01CD5678) were parked in the basement parking of Block B. Both vehicles are completely gutted - total loss. The fire started in the electrical room on the ground floor around 2 PM and spread rapidly. I was in a meeting on the 3rd floor when the alarm went off. By the time fire services arrived the basement was an inferno. I have the fire department incident report and CCTV footage showing the timeline. Please process urgently as I need transport for my team.',
 '2025-01-15', 4500000, 'AUTO', 'Block B Basement Parking, Tech Park, Manapakkam, Chennai'),

('CLM-001-PROP', 'POL-PROP-001', 'CUST001',
 'Our entire office on the 3rd floor of Block B, Manapakkam Tech Park has been destroyed in the fire on January 15th. All server rooms, workstations (150+ machines), furniture, and documents are gone. The structural engineer says the floor is compromised and cannot be reoccupied. We have the fire department report, structural assessment, and our asset inventory. The preliminary estimate from our contractor Landmark Constructions is Rs 3.2 crore for rebuild and equipment replacement. We need emergency workspace arrangements for 250 employees immediately.',
 '2025-01-15', 32000000, 'PROPERTY', '3rd Floor, Block B, Tech Park, Manapakkam, Chennai'),

('CLM-001-WC', 'POL-WC-001', 'CUST001',
 'Three of my employees were hospitalized due to the Manapakkam Tech Park fire on January 15th. Suresh Kumar (EMP-101) suffered second-degree burns on arms and face while helping evacuate colleagues. Lakshmi Devi (EMP-205) inhaled heavy smoke and is in ICU with respiratory distress. Ravi Shankar (EMP-089) fell during evacuation and fractured his right leg. All three are at Apollo Hospital, Guindy. Medical bills are mounting - Suresh alone is at Rs 8 lakhs already. I need immediate claim processing and direct hospital settlement if possible.',
 '2025-01-15', 2500000, 'WORKERS_COMP', 'Block B, Tech Park, Manapakkam + Apollo Hospital, Guindy'),

-- CUST002: Legitimate property claim (SME)
('CLM-002-PROP', 'POL-PROP-002', 'CUST002',
 'Our startup office at 15 Innovation Drive was affected by the Manapakkam fire. While our building did not catch fire directly, the adjacent Block B collapse caused debris to crash through our windows and roof. We have water damage from the fire department hoses. Our server rack (8 servers), 20 MacBooks, and all office furniture are damaged. Contractor estimate for repairs is Rs 18 lakhs. We had just signed a Series A and cannot afford this setback. Photos and contractor report attached.',
 '2025-01-15', 1800000, 'PROPERTY', '15 Innovation Drive, adjacent to Block B, Manapakkam'),

-- CUST003: Standard auto claim (parked near the area)
('CLM-003-AUTO', 'POL-AUTO-003', 'CUST003',
 'My Hyundai i20 was parked on the street near Manapakkam Tech Park when the fire happened on Jan 15. Falling debris hit the car and the heat cracked the windshield and melted the rear bumper. Garage estimate is Rs 1.5 lakhs for repairs. I have photos and the garage assessment from Sri Balaji Motors.',
 '2025-01-15', 150000, 'AUTO', 'Street parking, Manapakkam Road, near Tech Park'),

-- CUST005: SUSPICIOUS claim (new customer, inflated, vague)
('CLM-005-AUTO', 'POL-AUTO-005', 'CUST005',
 'My car was damaged badly in the Manapakkam fire area. Everything is destroyed. I need Rs 2.8 lakhs for the damage. The car was parked somewhere near the tech park. I dont remember exactly where. It happened around the same time as the big fire. I dont have photos because my phone was also damaged. The garage says it will cost a lot. Please process quickly I need the money urgently for personal emergency.',
 '2025-01-15', 280000, 'AUTO', 'Near Manapakkam Tech Park area');

-- Insert Providers
INSERT INTO RAW.PROVIDERS VALUES
('PROV-001', 'Apollo Hospital Guindy', 'HOSPITAL', 'WORKERS_COMP', 'IN_NETWORK', FALSE, 350000, 1200, 0, 'Chennai', 'Tamil Nadu', 'MH-TN-2345', CURRENT_TIMESTAMP()),
('PROV-002', 'Sri Balaji Motors', 'GARAGE', 'AUTO', 'IN_NETWORK', FALSE, 85000, 800, 0, 'Chennai', 'Tamil Nadu', 'GA-TN-5678', CURRENT_TIMESTAMP()),
('PROV-003', 'Landmark Constructions', 'CONTRACTOR', 'PROPERTY', 'IN_NETWORK', FALSE, 5000000, 150, 0, 'Chennai', 'Tamil Nadu', 'CN-TN-9012', CURRENT_TIMESTAMP()),
('PROV-004', 'QuickFix Garage', 'GARAGE', 'AUTO', 'OUT_NETWORK', TRUE, 250000, 50, 8, 'Chennai', 'Tamil Nadu', 'GA-TN-0000', CURRENT_TIMESTAMP()),
('PROV-005', 'Shady Repairs Unlimited', 'GARAGE', 'AUTO', 'OUT_NETWORK', TRUE, 400000, 20, 15, 'Chennai', 'Tamil Nadu', 'GA-TN-FAKE', CURRENT_TIMESTAMP());

-- Insert Fraud Indicators (historical patterns for RAG)
-- Using INSERT INTO ... SELECT for PARSE_JSON compatibility

INSERT INTO RAW.FRAUD_INDICATORS SELECT
 'FRD-001', 'DISASTER_OPPORTUNISM', 'Claims submitted immediately after a publicized disaster event by customers with no geographic or temporal connection to the incident. Key signals: new policy (<6 months), no prior claims, vague location details, inflated amounts relative to coverage, urgency pressure language ("need money urgently", "personal emergency"). Often uses generic descriptions without specific details about the actual incident.',
 'AUTO', 'HIGH', PARSE_JSON('{"signals": ["new_policy", "vague_location", "urgency_language", "no_photos", "inflated_amount"]}'),
 'My car was damaged in the fire area. Everything destroyed. Need money urgently for personal emergency. Dont have photos. Car was somewhere near the incident.',
 45, '2024-08-15'::DATE, CURRENT_TIMESTAMP();

INSERT INTO RAW.FRAUD_INDICATORS SELECT
 'FRD-002', 'DUPLICATE_SUBMISSION', 'Same incident claimed multiple times across different policy numbers or slightly different descriptions. Key signals: same incident date, same customer, similar claim text, overlapping coverage. Note: legitimate multi-LOB claims from genuine disasters are NOT duplicates - they cover different assets under different policies.',
 'AUTO', 'MEDIUM', PARSE_JSON('{"signals": ["same_date", "similar_text", "overlapping_coverage", "same_damage_description"]}'),
 'Two claims for same vehicle damage submitted under different policy numbers with slightly altered descriptions.',
 120, '2024-12-01'::DATE, CURRENT_TIMESTAMP();

INSERT INTO RAW.FRAUD_INDICATORS SELECT
 'FRD-003', 'INFLATED_BILLING', 'Claim amounts significantly exceeding the actual value of the damaged asset or market rates for repairs. Key signals: claim amount > 80% of coverage limit on low-value asset, amount exceeds market value, unfamiliar provider, no supporting documentation from recognized providers.',
 'AUTO', 'HIGH', PARSE_JSON('{"signals": ["amount_vs_asset_value", "exceeds_market_rate", "unrecognized_provider", "no_docs"]}'),
 'Claiming Rs 2.8 lakhs damage on a 2018 Maruti Alto valued at Rs 3 lakhs. Amount is 93% of vehicle value for non-total-loss claim.',
 89, '2024-11-20'::DATE, CURRENT_TIMESTAMP();

INSERT INTO RAW.FRAUD_INDICATORS SELECT
 'FRD-004', 'STAGED_ACCIDENT', 'Accident details that dont match physical evidence or witness accounts. Common in auto claims. Key signals: inconsistent damage patterns, no independent witnesses, provider known for fraud, excessive treatment for minor injury.',
 'AUTO', 'CRITICAL', PARSE_JSON('{"signals": ["inconsistent_damage", "no_witnesses", "flagged_provider", "excessive_treatment"]}'),
 'Vehicle shows front-end damage but claim describes rear collision. No police report filed. Garage previously flagged.',
 32, '2024-10-05'::DATE, CURRENT_TIMESTAMP();

INSERT INTO RAW.FRAUD_INDICATORS SELECT
 'FRD-005', 'EMOTIONAL_MANIPULATION', 'Claim narratives using excessive emotional language, urgency, personal hardship stories to pressure fast approval without documentation. Key signals: urgency phrases, hardship narrative, resistance to documentation requests, threats.',
 'AUTO', 'MEDIUM', PARSE_JSON('{"signals": ["urgency_phrases", "hardship_narrative", "avoids_documentation", "pressure_tactics"]}'),
 'Please process immediately, I am in dire need. My family depends on this. I cannot provide documents right now due to personal crisis. Just approve it.',
 67, '2024-09-12'::DATE, CURRENT_TIMESTAMP();

-- Insert Interactions (including distressed calls from HNW client)
INSERT INTO RAW.INTERACTIONS VALUES
('INT-001', 'CUST001', 'CALL', '2025-01-15 15:30:00',
 'Call transcript: Customer is extremely distressed. Speaking rapidly. "This is Rajesh Krishnamurthy, policy holder. There has been a massive fire at Manapakkam Tech Park. My entire office is destroyed. My cars are gone. Three of my employees are in hospital. I need someone from your company here NOW. This is a catastrophe. I have been a loyal customer for 10 years paying Rs 1.8 lakhs annual premium. I expect priority treatment. My staff are suffering. The fire department is still on scene. I will send all documents but I need immediate acknowledgment that my claims will be fast-tracked. This is not a normal situation."',
 'INBOUND', 'CLAIM_EMERGENCY', 'ESCALATED', 'AGT-005', 420, NULL, CURRENT_TIMESTAMP()),

('INT-002', 'CUST001', 'CALL', '2025-01-16 09:00:00',
 'Call transcript: Follow-up call next day. Customer calmer but still stressed. "Good morning, this is Rajesh again regarding the Manapakkam fire claims. I have three claims submitted yesterday - auto, property, and workers comp. I need a single point of contact. I cannot be explaining the same disaster to three different departments. My employees need medical attention and the hospital is asking for payment guarantee. Can you assign a dedicated claims manager? Also, I noticed competitor XYZ Insurance is advertising faster claims settlement. I have been considering moving my portfolio if this is not handled properly."',
 'INBOUND', 'CLAIM_FOLLOWUP', 'PENDING', 'AGT-005', 300, 4, CURRENT_TIMESTAMP()),

('INT-003', 'CUST004', 'CALL', '2025-01-15 16:00:00',
 'Call transcript: "This is Meenakshi Rajan from Chennai Manufacturing Corp. Our factory in Manapakkam Industrial Estate has been affected by the fire spreading from the tech park. Half our production floor is damaged and 12 workers are injured. We need emergency claims processing. Our policy numbers are POL-PROP-004 and POL-WC-004. The fire department report will be available tomorrow. We have CCTV footage. Please expedite."',
 'INBOUND', 'CLAIM_EMERGENCY', 'ESCALATED', 'AGT-007', 280, NULL, CURRENT_TIMESTAMP()),

('INT-004', 'CUST003', 'EMAIL', '2025-01-16 11:00:00',
 'Subject: Claim for car damage - Manapakkam fire. Hi, I am writing to follow up on my claim CLM-003-AUTO submitted yesterday. My car was parked on the road near the tech park. The garage has given me a detailed estimate. When can I expect the surveyor to visit? I am managing with public transport for now. Thanks, Arjun.',
 'INBOUND', 'CLAIM_FOLLOWUP', 'RESOLVED', 'AGT-003', 0, 7, CURRENT_TIMESTAMP()),

('INT-005', 'CUST002', 'CHAT', '2025-01-17 14:00:00',
 'Chat transcript: Customer: "Hi, any update on my property claim? Its been 2 days." Agent: "Your claim CLM-002-PROP is being processed. Surveyor scheduled for tomorrow." Customer: "OK thanks. We are operating from a co-working space currently. The team morale is low. We just raised Series A funding and this fire is very bad timing. Hope the settlement is quick." Agent: "We understand the urgency. Will prioritize."',
 'INBOUND', 'CLAIM_STATUS', 'RESOLVED', 'AGT-002', 0, 6, CURRENT_TIMESTAMP());

-- Insert Payments (showing payment patterns)
INSERT INTO RAW.PAYMENTS VALUES
-- CUST001: Always pays on time (loyal HNW)
('PAY-001-01', 'POL-AUTO-001', 'CUST001', '2024-01-05', '2024-01-10', 45000, 'PAID', 'AUTO_DEBIT', 0, CURRENT_TIMESTAMP()),
('PAY-001-02', 'POL-PROP-001', 'CUST001', '2024-01-05', '2024-01-10', 95000, 'PAID', 'NEFT', 0, CURRENT_TIMESTAMP()),
('PAY-001-03', 'POL-WC-001', 'CUST001', '2024-01-05', '2024-01-10', 40000, 'PAID', 'NEFT', 0, CURRENT_TIMESTAMP()),
-- CUST003: Occasionally late
('PAY-003-01', 'POL-AUTO-003', 'CUST003', '2024-03-20', '2024-03-10', 12000, 'PAID', 'UPI', 10, CURRENT_TIMESTAMP()),
-- CUST005: Just started, one payment
('PAY-005-01', 'POL-AUTO-005', 'CUST005', '2024-11-05', '2024-11-05', 8000, 'PAID', 'CARD', 0, CURRENT_TIMESTAMP());

-- Insert Underwriting Guidelines (for RAG)
INSERT INTO RAW.UNDERWRITING_GUIDELINES VALUES
('UG-001', 'AUTO', 'Vehicle Age and Condition', 'Vehicles older than 10 years require physical inspection before coverage. Depreciation applies at 10% per year after first 3 years. Total coverage cannot exceed IDV (Insured Declared Value). High-performance vehicles (>200HP) require additional premium of 15%. Electric vehicles get 5% discount.',
 'ALL', 'IRDAI-Motor-2023', '2023-04-01', 'CURRENT'),
('UG-002', 'PROPERTY', 'Commercial Property Fire Coverage', 'Fire coverage for commercial properties must include fire brigade charges, debris removal, and architects fees (up to 5% of claim). Electrical fire caused by faulty wiring - covered. Arson by owner - excluded. Sum insured must reflect current replacement value, not market value. Annual escalation of 8% recommended for commercial.',
 'ALL', 'IRDAI-Fire-2023', '2023-04-01', 'CURRENT'),
('UG-003', 'WORKERS_COMP', 'Workplace Injury Classification', 'Injuries classified as: Minor (first aid only, no leave), Moderate (medical attention, <7 days leave), Severe (hospitalization, >7 days), Critical (life-threatening, ICU). Compensation: Minor=medical only, Moderate=medical+50% wages, Severe=medical+75% wages+disability, Critical=medical+100% wages+disability+family support.',
 'ALL', 'IRDAI-WC-2023', '2023-04-01', 'CURRENT'),
('UG-004', 'PROPERTY', 'High-Value Property Risk Assessment', 'Properties with sum insured >Rs 5 crore require: (1) Fire safety audit, (2) Sprinkler system verification, (3) Electrical safety certificate <6 months old, (4) Building stability certificate. Risk tier: LOW if all 4 present, MEDIUM if 2-3, HIGH if 0-1. Premium multiplier: LOW=1.0x, MEDIUM=1.3x, HIGH=1.8x.',
 'HIGH', 'IRDAI-Property-2023', '2023-04-01', 'CURRENT'),
('UG-005', 'AUTO', 'Fleet Insurance Underwriting', 'Fleet of 3+ vehicles: bundle discount 10%. Fleet of 10+: 15%. All vehicles must be registered to same owner/company. Mix of personal and commercial use disqualifies fleet discount. GPS tracking installed: additional 5% discount. No-claim bonus applies per vehicle, not fleet-wide.',
 'LOW', 'IRDAI-Motor-2023', '2023-04-01', 'CURRENT');

-- Insert Retention Playbooks (for Churn RAG)
-- Using INSERT INTO ... SELECT for PARSE_JSON compatibility

INSERT INTO RAW.RETENTION_PLAYBOOKS SELECT
 'PB-001', 'HNW', 'poor_claims_experience', 'VIP Claims Recovery Playbook',
 'For HNW customers showing dissatisfaction after a claims experience: (1) Assign dedicated senior claims manager within 4 hours, (2) Provide direct mobile number for updates, (3) Fast-track remaining claims to 48-hour resolution, (4) Schedule executive callback from branch head within 24 hours, (5) Offer complimentary coverage enhancement for next renewal period, (6) Document service recovery in CRM for renewal team.',
 PARSE_JSON('["dedicated_manager", "fast_track_claims", "executive_callback", "coverage_enhancement"]'), 0.78, 'ALL';

INSERT INTO RAW.RETENTION_PLAYBOOKS SELECT
 'PB-002', 'STANDARD', 'price_sensitivity', 'Standard Retention - Price Sensitive',
 'For standard customers citing pricing concerns: (1) Review current premium vs market benchmark, (2) Offer 10% loyalty discount if tenure >2 years, (3) Suggest deductible increase for premium reduction, (4) Bundle offer: add second LOB for 15% total discount, (5) Highlight no-claim bonus accrued. Do NOT offer more than 15% total discount - refer to manager if needed.',
 PARSE_JSON('["loyalty_discount", "deductible_adjustment", "bundle_offer", "ncb_highlight"]'), 0.55, 'ALL';

INSERT INTO RAW.RETENTION_PLAYBOOKS SELECT
 'PB-003', 'HNW', 'competitor_offer', 'HNW Counter-Competitive Playbook',
 'When HNW customer mentions competitor: (1) Never disparage competitor, (2) Emphasize relationship value and claims track record, (3) Offer dedicated relationship manager (if not already assigned), (4) Match or beat competitor premium if within 5% (pre-approved), (5) Offer premium lock for 3 years, (6) Highlight multi-LOB integration advantage competitors cannot match.',
 PARSE_JSON('["premium_match", "relationship_manager", "premium_lock_3yr", "multi_lob_advantage"]'), 0.72, 'ALL';

INSERT INTO RAW.RETENTION_PLAYBOOKS SELECT
 'PB-004', 'SME', 'poor_claims_experience', 'SME Claims Recovery',
 'For SME customers with bad claims experience: (1) Acknowledge the experience explicitly, (2) Assign business insurance specialist, (3) Expedite any open claims, (4) Offer business continuity consultation (free), (5) 5% renewal discount as goodwill, (6) Quarterly check-in calls scheduled.',
 PARSE_JSON('["acknowledge", "specialist_assignment", "expedite_claims", "business_continuity", "renewal_discount"]'), 0.62, 'ALL';

-- Insert Actuarial Tables
INSERT INTO RAW.ACTUARIAL_TABLES VALUES
('ACT-001', 'AUTO', 'LOW', 'CHENNAI_GENERAL', 0.035, 1.0, 1.1, 1.0, '2024-01-01', '2025-12-31'),
('ACT-002', 'AUTO', 'MEDIUM', 'CHENNAI_GENERAL', 0.035, 1.3, 1.1, 1.2, '2024-01-01', '2025-12-31'),
('ACT-003', 'AUTO', 'HIGH', 'CHENNAI_GENERAL', 0.035, 1.8, 1.1, 1.5, '2024-01-01', '2025-12-31'),
('ACT-004', 'PROPERTY', 'LOW', 'CHENNAI_MANAPAKKAM', 0.0025, 1.0, 1.3, 1.0, '2024-01-01', '2025-12-31'),
('ACT-005', 'PROPERTY', 'MEDIUM', 'CHENNAI_MANAPAKKAM', 0.0025, 1.4, 1.3, 1.3, '2024-01-01', '2025-12-31'),
('ACT-006', 'PROPERTY', 'HIGH', 'CHENNAI_MANAPAKKAM', 0.0025, 2.0, 1.3, 1.6, '2024-01-01', '2025-12-31'),
('ACT-007', 'WORKERS_COMP', 'LOW', 'CHENNAI_MANAPAKKAM', 0.02, 1.0, 1.0, 1.0, '2024-01-01', '2025-12-31'),
('ACT-008', 'WORKERS_COMP', 'MEDIUM', 'CHENNAI_MANAPAKKAM', 0.02, 1.3, 1.0, 1.2, '2024-01-01', '2025-12-31');

## DONE - Raw tables created with Manapakkam fire sample data